# Applio CEVC Roughness Adapter — живая консоль

Этот Colab запускает обычный веб-интерфейс Applio в переднем плане. Ячейка запуска остаётся активной и показывает весь вывод.

Порядок: Google Drive → установка → подключение моделей → запуск. В интерфейсе: Preprocess → Extract Features → Check CEVC Data → Train Roughness Adapter.


In [ ]:
# @title 1. Подключить Google Drive
from google.colab import drive
from google.colab._message import MessageError

try:
    drive.mount("/content/drive")
except MessageError as error:
    print(f"Google Drive не подключён: {error}")


In [ ]:
# @title 2. Установить Applio CEVC
from pathlib import Path
import shutil
import subprocess
import sys

APP_DIR = Path("/content/Applio")
REPO_URL = "https://github.com/egor125552/Applio.git"
REPO_REF = "agent/compact-expressive-vc-architecture"
NOTEBOOK_BUILD = "cevc-adapter-v4-foreground-public-url"

def run(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

if APP_DIR.exists():
    shutil.rmtree(APP_DIR)

run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, APP_DIR])
run(["apt-get", "update", "-y"])
run(["apt-get", "install", "-y", "ffmpeg", "portaudio19-dev", "libsndfile1", "util-linux"])
run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])

uv = shutil.which("uv") or "/root/.local/bin/uv"
run([
    uv, "pip", "install", "--system", "-r", "requirements.txt",
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
    "--index-strategy", "unsafe-best-match",
], cwd=APP_DIR)
run([
    sys.executable, "-u", "core.py", "prerequisites",
    "--models", "True", "--pretraineds_hifigan", "True", "--exe", "False",
], cwd=APP_DIR)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=APP_DIR, text=True).strip()
print(f"Готово. Сборка: {NOTEBOOK_BUILD}")
print(f"Ветка: {REPO_REF}")
print(f"Коммит: {commit}")


In [ ]:
# @title 3. Подключить модели с Google Drive
from pathlib import Path

APP_DIR = Path("/content/Applio")
LOCAL_LOGS = APP_DIR / "logs"
DRIVE_BACKUP = Path("/content/drive/MyDrive/ApplioBackup")
LOCAL_LOGS.mkdir(parents=True, exist_ok=True)

if Path("/content/drive").is_mount():
    DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
    linked = 0
    for item in DRIVE_BACKUP.iterdir():
        target = LOCAL_LOGS / item.name
        if target.exists() or target.is_symlink():
            continue
        target.symlink_to(item, target_is_directory=item.is_dir())
        linked += 1
    print(f"Подключено объектов: {linked}")
    print(f"Постоянная папка моделей: {DRIVE_BACKUP}")
else:
    print("Google Drive не подключён; данные сохранятся только до сброса Colab.")


In [ ]:
# @title 4. Запустить Applio — передний план, публичная ссылка и живая консоль
from IPython.display import HTML, display
from pathlib import Path
import codecs
import os
import re
import shlex
import shutil
import subprocess
import sys

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
APP_LOG = DIAG_DIR / "app.log"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, "-u", "app.py",
    "--server-name", "0.0.0.0", "--port", "6969", "--share",
]
environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["GRADIO_ANALYTICS_ENABLED"] = "False"

script_executable = shutil.which("script")
wrapped_command = (
    [script_executable, "-q", "-f", "-e", "-c", shlex.join(command), "/dev/null"]
    if script_executable else command
)

print("Applio запускается в ПЕРЕДНЕМ ПЛАНЕ.", flush=True)
print("Ячейка останется активной; здесь будут все этапы, проценты, loss и ошибки.", flush=True)
print("Ожидаю публичную ссылку Gradio…", flush=True)

process = subprocess.Popen(
    wrapped_command,
    cwd=APP_DIR,
    env=environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)

url_pattern = re.compile(rb"https://[a-zA-Z0-9-]+\.gradio\.live")
seen_url = None
scan_buffer = b""
decoder = codecs.getincrementaldecoder("utf-8")("replace")

with APP_LOG.open("wb", buffering=0) as log_file:
    while True:
        chunk = os.read(process.stdout.fileno(), 1024)
        if not chunk:
            break
        log_file.write(chunk)
        text = decoder.decode(chunk)
        if text:
            sys.stdout.write(text)
            sys.stdout.flush()
        scan_buffer = (scan_buffer + chunk)[-4096:]
        match = url_pattern.search(scan_buffer)
        if match:
            url = match.group(0).decode("utf-8")
            if url != seen_url:
                seen_url = url
                print(f"\nПУБЛИЧНАЯ ССЫЛКА: {url}", flush=True)
                display(HTML(f'<p><b>Открыть Applio:</b> <a href="{url}" target="_blank">{url}</a></p>'))

tail = decoder.decode(b"", final=True)
if tail:
    sys.stdout.write(tail)
    sys.stdout.flush()

returncode = process.wait()
if returncode != 0:
    raise RuntimeError(
        f"Applio завершился с кодом {returncode}. Точная ошибка находится выше и в {APP_LOG}."
    )
print("Applio завершился штатно.", flush=True)


In [ ]:
# @title 5. Скачать технические логи после остановки Applio
from datetime import datetime, timezone
from google.colab import files
from pathlib import Path
import zipfile

APP_DIR = Path("/content/Applio")
DIAG_DIR = APP_DIR / "cevc_diagnostics"
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
archive = Path(f"/content/CEVC_log_{timestamp}.zip")

with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    if DIAG_DIR.exists():
        for path in sorted(DIAG_DIR.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=f"diagnostics/{path.relative_to(DIAG_DIR)}")
    logs_dir = APP_DIR / "logs"
    if logs_dir.exists():
        useful_names = {
            "model_info.json", "filelist.txt", "cevc_source_manifest.json",
            "cevc_expressive_manifest.json", "training_history.json",
        }
        for path in sorted(logs_dir.rglob("*")):
            if path.is_file() and path.name in useful_names:
                bundle.write(path, arcname=f"experiments/{path.relative_to(logs_dir)}")

print(f"Готово: {archive}")
files.download(str(archive))
